# 🦙 Fine-Tuning Llama 2 with QLoRA on Google Colab

**Objective:** Fine-tune the `NousResearch/Llama-2-7b-chat-hf` model on the `mlabonne/guanaco-llama2-1k` instruction dataset using **QLoRA** (Quantized Low-Rank Adaptation), a parameter-efficient fine-tuning (PEFT) technique optimized for consumer-grade GPUs like the T4.

### Why QLoRA?
Full fine-tuning of a 7B parameter model requires ~28GB+ of VRAM just for the weights in FP32. Google Colab’s free T4 GPU provides only **~15GB VRAM**. QLoRA solves this by:
1.  **Quantizing** the base model to **4-bit precision** (NF4), reducing its memory footprint by ~4x.
2.  Injecting small, trainable **LoRA adapters** into the frozen base model (only ~0.1% of parameters are trained).
3.  Using **double quantization** and **paged optimizers** to further reduce memory usage.

### Notebook Phases
| Phase | Description |
|---|---|
| **Phase 1** | Environment Setup (Drive, Secrets, Libraries) |
| **Phase 2** | Model & Tokenizer Initialization (4-bit Quantization, LoRA) |
| **Phase 3** | Dataset Preparation (Loading & Formatting) |
| **Phase 4** | Training Configuration (`TrainingArguments` & `SFTTrainer`) |
| **Phase 5** | Inference & Testing (Generating with the Fine-Tuned Model) |

---
## Phase 1: Environment Setup
This phase prepares the Colab runtime with all necessary dependencies and credentials.

### 📖 Guide: Google Drive Mounting

We mount Google Drive to provide a **persistent storage location** for saving model checkpoints and training logs. Without this, all outputs are lost when the Colab session ends. This is especially critical for long training runs that might time out.

In [ ]:
# --- Google Drive Mounting ---
# Mount Google Drive to persist checkpoints and training artifacts across sessions.
from google.colab import drive  # Import the Colab drive utility
drive.mount('/content/drive')    # Mount the user's Drive at /content/drive

### 📖 Guide: Hugging Face Hub Authentication

Many models on the Hugging Face Hub (especially Llama 2) are **gated**: you must accept a license agreement before downloading them. The `HF_TOKEN` is required to authenticate your identity.

**Security Best Practice:** We use `google.colab.userdata.get('HF_TOKEN')` to retrieve the token from Colab’s built-in **Secrets Manager** (the 🔑 icon in the left sidebar). This avoids hardcoding tokens in the notebook, which would be a major security risk if shared.

In [ ]:
# --- Hugging Face Hub Authentication ---
from google.colab import userdata       # Import Colab's secure secrets manager
from huggingface_hub import login        # Import the HF Hub login function

# Retrieve the Hugging Face token securely from Colab Secrets (Key icon in sidebar).
# To set this up: Go to Colab sidebar > Secrets > Add 'HF_TOKEN' with your token.
hf_token = userdata.get('HF_TOKEN')     # Fetch token without exposing it in code
login(token=hf_token)                    # Authenticate with the Hugging Face Hub

### 📖 Guide: Library Installation

We install the core libraries required for the QLoRA fine-tuning pipeline:

| Library | Purpose |
|---|---|
| `transformers` | Provides the model architecture (`AutoModelForCausalLM`) and tokenizer. |
| `datasets` | Efficient loading and processing of training data from Hugging Face Hub. |
| `peft` | Implements **LoRA/QLoRA** — the parameter-efficient fine-tuning method. |
| `trl` | Provides `SFTTrainer`, a high-level trainer for Supervised Fine-Tuning. |
| `bitsandbytes` | Enables **4-bit quantization** via `BitsAndBytesConfig` for memory efficiency. |
| `accelerate` | Handles device placement and mixed-precision training orchestration. |

> **Note:** We pin compatible versions to avoid breaking changes between libraries.

In [ ]:
# --- Install Required Libraries ---
# -q: quiet mode to reduce installation output noise.
# -U: upgrade to latest compatible version.
# These libraries form the core QLoRA fine-tuning stack.
!pip install -qU \
    transformers \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate

### 📖 Guide: Importing the Python Modules

After installation, we import all necessary Python modules. Organizing imports at the top of the pipeline improves readability and ensures all dependencies are available before any computation begins.

In [ ]:
# --- Import All Required Modules ---
import os                                        # For environment variable access
import torch                                     # PyTorch: the deep learning backend
import gc                                        # Garbage collector for manual VRAM cleanup
from datasets import load_dataset                # Load datasets from the Hugging Face Hub
from transformers import (                       # Core Hugging Face transformers components
    AutoModelForCausalLM,                        #   Auto-detect and load causal language models
    AutoTokenizer,                               #   Auto-detect and load the matching tokenizer
    BitsAndBytesConfig,                          #   Configure 4-bit/8-bit quantization
    TrainingArguments,                           #   Define hyperparameters for the Trainer
    pipeline,                                    #   High-level inference pipeline
    logging,                                     #   Control transformers library log verbosity
)
from peft import LoraConfig, PeftModel           # LoRA adapter configuration and model wrapper
from trl import SFTTrainer                       # Supervised Fine-Tuning Trainer from TRL

---
## Phase 2: Model & Tokenizer Initialization
This phase loads the pre-trained Llama 2 model in 4-bit precision and configures the LoRA adapters.

### 📖 Guide: Defining Core Identifiers

We define the model ID, dataset name, and the output name for our fine-tuned model upfront. Using `NousResearch/Llama-2-7b-chat-hf` — this is a community-hosted mirror of Meta’s Llama 2 7B Chat model, which is pre-trained for dialogue and follows the **Llama 2 chat prompt template**:

```
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST] {model_reply} </s>
```

The dataset `mlabonne/guanaco-llama2-1k` is a **pre-formatted** subset (1,000 samples) of the OpenAssistant Guanaco dataset, already structured to match this template.

In [ ]:
# --- Core Identifiers ---
# The base model to fine-tune (Llama 2 7B Chat variant from NousResearch mirror)
MODEL_ID = "NousResearch/Llama-2-7b-chat-hf"

# The instruction-following dataset, pre-formatted for the Llama 2 chat template
DATASET_NAME = "mlabonne/guanaco-llama2-1k"

# Name for the fine-tuned LoRA adapter output
NEW_MODEL_NAME = "Llama-2-7b-chat-finetune"

### 📖 Guide: 4-Bit Quantization with `BitsAndBytesConfig`

**Why Quantization?** The Llama 2 7B model has ~7 billion parameters. In FP16 (2 bytes/param), that’s ~14GB — nearly filling the T4’s 15GB VRAM with no room left for gradients, optimizer states, or activations.

**4-bit quantization** compresses each weight to ~0.5 bytes, reducing the model footprint to ~3.5GB and freeing VRAM for training.

| Parameter | Value | Explanation |
|---|---|---|
| `load_in_4bit` | `True` | Enable 4-bit weight quantization. |
| `bnb_4bit_quant_type` | `"nf4"` | **NormalFloat4** — an information-theoretically optimal quantization type for normally distributed weights. Outperforms standard FP4. |
| `bnb_4bit_compute_dtype` | `torch.float16` | Computation dtype during forward/backward passes. FP16 is optimal for T4 GPUs. |
| `bnb_4bit_use_double_quant` | `True` | **Double quantization** — quantizes the quantization constants themselves, saving an additional ~0.4 bits/param (~0.37GB for a 7B model). |

In [ ]:
# --- 4-Bit Quantization Configuration ---
# BitsAndBytesConfig defines how the model weights are quantized during loading.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                           # Load model weights in 4-bit precision
    bnb_4bit_quant_type="nf4",                   # NormalFloat4: optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16,        # Use FP16 for compute (optimal for T4 GPU)
    bnb_4bit_use_double_quant=True,              # Double quantization: quantize the quantization constants
)

### 📖 Guide: Loading the Base Model

We load the Llama 2 model with our quantization config. Key settings:
- **`device_map="auto"`**: Automatically distributes model layers across available GPUs/CPU. On a single T4, this places everything on GPU 0.
- **`use_cache = False`**: Disables KV-cache (used during inference for speed). During training, it’s incompatible with gradient checkpointing and wastes VRAM.
- **`pretraining_tp = 1`**: Sets tensor parallelism to 1 (single GPU). This flag exists because the original Llama 2 was trained with tensor parallelism; setting it to 1 ensures correct behavior on a single device.
- **`attn_implementation="eager"`**: Uses the standard attention implementation. On T4 GPUs that lack FlashAttention2 support, this is the reliable choice.

In [ ]:
# --- Load Base Model with 4-Bit Quantization ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,                                    # Hugging Face model identifier
    quantization_config=bnb_config,              # Apply 4-bit quantization from above
    device_map="auto",                           # Automatically place model on available GPU(s)
    attn_implementation="eager",                 # Standard attention (T4 lacks FlashAttention2 support)
)

# Disable KV-cache: incompatible with gradient checkpointing during training
model.config.use_cache = False

# Set tensor parallelism degree to 1 (single GPU training)
model.config.pretraining_tp = 1

# Print model size confirmation
print(f"\n\u2705 Model loaded: {MODEL_ID}")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Device: {model.device}")

### 📖 Guide: Loading the Tokenizer

The tokenizer converts text into token IDs that the model understands, and vice versa.

**Important configurations:**
- **`pad_token = eos_token`**: Llama 2’s tokenizer has no dedicated padding token. We reuse the End-of-Sequence (EOS) token for padding. This is standard practice for decoder-only models.
- **`padding_side = "right"`**: We pad sequences on the right side. For causal (left-to-right) language models, right-padding ensures the model attends to actual tokens first, which prevents numerical instability during FP16 training.

In [ ]:
# --- Load Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,                                    # Must match the model being fine-tuned
    trust_remote_code=True,                      # Allow execution of custom tokenizer code if needed
)

# Set padding token to EOS token (Llama 2 has no dedicated pad token)
tokenizer.pad_token = tokenizer.eos_token

# Pad on the right side to avoid FP16 numerical issues in causal attention
tokenizer.padding_side = "right"

print(f"\n\u2705 Tokenizer loaded.")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")
print(f"   Pad token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

### 📖 Guide: LoRA Configuration

**LoRA (Low-Rank Adaptation)** is the core of QLoRA. Instead of updating all 7B parameters, LoRA injects **small, trainable low-rank matrices** into specific layers. This reduces trainable parameters from ~7B to ~33M (~0.5%).

**How it works:**
For a weight matrix `W` of dimension `d × d`, LoRA decomposes the update as `ΔW = A × B`, where `A` is `d × r` and `B` is `r × d`. With `r = 64` (rank), this is far fewer parameters than the full `d × d` update.

| Parameter | Value | Explanation |
|---|---|---|
| `r` | `64` | **Rank** of the LoRA decomposition. Higher = more expressive but more VRAM. 64 is a strong default for 7B models. |
| `lora_alpha` | `16` | **Scaling factor**. The LoRA update is scaled by `alpha/r`. Lower alpha relative to r produces more conservative updates. |
| `lora_dropout` | `0.1` | Dropout on LoRA layers to prevent overfitting (10% of activations are randomly zeroed). |
| `bias` | `"none"` | Don’t add LoRA to bias terms (standard practice — biases are small and fast to train). |
| `task_type` | `"CAUSAL_LM"` | Indicates this is causal (autoregressive) language modeling, ensuring correct loss computation. |

In [ ]:
# --- LoRA (Low-Rank Adaptation) Configuration ---
peft_config = LoraConfig(
    r=64,                                        # Rank of LoRA decomposition matrices (expressiveness)
    lora_alpha=16,                               # Scaling factor: update scaled by alpha/r
    lora_dropout=0.1,                            # 10% dropout on LoRA layers to reduce overfitting
    bias="none",                                 # Don't apply LoRA to bias terms (not needed)
    task_type="CAUSAL_LM",                       # Task type: causal (autoregressive) language modeling
)

print(f"\n\u2705 LoRA Config: rank={peft_config.r}, alpha={peft_config.lora_alpha}, dropout={peft_config.lora_dropout}")

---
## Phase 3: Dataset Preparation
This phase loads the instruction-following dataset and inspects it before training.

### 📖 Guide: Loading & Inspecting the Dataset

We use the **`mlabonne/guanaco-llama2-1k`** dataset, which is a 1,000-sample subset of the [OpenAssistant Guanaco](https://huggingface.co/datasets/timdettmers/openassistant-guanaco) dataset, **pre-formatted** to match the Llama 2 chat prompt template.

**Why a pre-formatted dataset?**
- Llama 2 Chat models expect a **specific prompt structure** with `[INST]` and `[/INST]` delimiters.
- Using a pre-formatted dataset eliminates the need for a custom formatting function.
- The `SFTTrainer` will directly use the `"text"` field from this dataset.

**Dataset sources:**
- Original: [timdettmers/openassistant-guanaco](https://huggingface.co/datasets/timdettmers/openassistant-guanaco)
- 1K reformatted: [mlabonne/guanaco-llama2-1k](https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k)
- Full reformatted: [mlabonne/guanaco-llama2](https://huggingface.co/datasets/mlabonne/guanaco-llama2)

In [ ]:
# --- Load the Instruction Dataset ---
# Load the 1K-sample Guanaco dataset, pre-formatted for Llama 2 chat template.
# split="train": load the training split (this dataset only has a train split).
dataset = load_dataset(DATASET_NAME, split="train")

# --- Inspect the Dataset ---
print(f"\n\u2705 Dataset loaded: {DATASET_NAME}")
print(f"   Number of samples: {len(dataset):,}")
print(f"   Features: {list(dataset.features.keys())}")
print(f"\n--- Sample Entry (first 500 chars) ---")
print(dataset[0]['text'][:500])                  # Preview the first sample

---
## Phase 4: Training Configuration
This phase sets up the training hyperparameters and launches the fine-tuning process.

### 📖 Guide: Training Arguments

The `TrainingArguments` class from Hugging Face defines **all hyperparameters** for the training loop. Here’s the rationale behind each setting:

| Parameter | Value | Why |
|---|---|---|
| `num_train_epochs` | `1` | 1 epoch is often sufficient for instruction tuning on small datasets. Prevents overfitting to the 1K samples. |
| `per_device_train_batch_size` | `4` | Batch size per GPU. 4 fits comfortably in T4 VRAM with 4-bit quantization. |
| `gradient_accumulation_steps` | `1` | Effective batch size = batch_size × accumulation_steps = 4. Increase if you want a larger effective batch without more VRAM. |
| `gradient_checkpointing` | `True` | **Critical for memory**: trades compute for VRAM by recomputing activations during backward pass instead of storing them. |
| `optim` | `"paged_adamw_32bit"` | **Paged optimizer**: automatically offloads optimizer states to CPU RAM when GPU VRAM is full. Prevents OOM crashes. |
| `learning_rate` | `2e-4` | Standard LR for QLoRA fine-tuning. |
| `lr_scheduler_type` | `"cosine"` | Cosine annealing: smoothly decays LR to near-zero, improving convergence. |
| `warmup_ratio` | `0.03` | Gradually ramp up LR for the first 3% of steps. Prevents early training instability. |
| `max_grad_norm` | `0.3` | Clip gradients to prevent exploding gradients (common in fine-tuning). |
| `weight_decay` | `0.001` | Light L2 regularization on all layers except bias/LayerNorm. |
| `fp16` | `True` | Enable FP16 mixed precision. **Optimal for T4** (set `bf16=True` instead on A100/H100). |
| `group_by_length` | `True` | Group similar-length sequences into batches to minimize padding waste. |
| `save_strategy` | `"epoch"` | Save a checkpoint at the end of each epoch. |

In [ ]:
# --- Training Hyperparameters ---
# Output directory for model checkpoints and TensorBoard logs
OUTPUT_DIR = "./results"                         # Checkpoints will be saved here

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,                       # Directory for checkpoints and logs
    num_train_epochs=1,                          # Train for 1 full epoch (sufficient for 1K instruction samples)
    per_device_train_batch_size=4,               # 4 samples per GPU per forward pass
    gradient_accumulation_steps=1,               # Effective batch size = 4 * 1 = 4
    gradient_checkpointing=True,                 # Trade compute for VRAM: recompute activations in backward pass
    optim="paged_adamw_32bit",                   # Paged optimizer: auto-offloads states to CPU if VRAM is full
    learning_rate=2e-4,                          # Standard learning rate for QLoRA
    lr_scheduler_type="cosine",                  # Cosine decay schedule for smooth convergence
    warmup_ratio=0.03,                           # Warm up LR for first 3% of training steps
    max_grad_norm=0.3,                           # Gradient clipping to prevent exploding gradients
    weight_decay=0.001,                          # Light L2 regularization on weights
    fp16=True,                                   # FP16 mixed precision (optimal for T4 GPU)
    bf16=False,                                  # Set True for A100/H100 GPUs (not supported on T4)
    group_by_length=True,                        # Group similar-length sequences to reduce padding waste
    logging_steps=25,                            # Log training metrics every 25 steps
    save_strategy="epoch",                       # Save checkpoint at end of each epoch
    report_to="tensorboard",                     # Send metrics to TensorBoard for visualization
)

### 📖 Guide: SFTTrainer Setup

The **`SFTTrainer`** from the `trl` library is a specialized trainer for **Supervised Fine-Tuning (SFT)**. It wraps the standard Hugging Face `Trainer` with additional features tailored for instruction tuning:

- **`dataset_text_field`**: Specifies which field in the dataset contains the training text. Our dataset uses `"text"`.
- **`max_seq_length`**: The maximum sequence length for training. We set this to `512` to balance context window and memory usage on T4.
- **`packing`**: When `False`, each sample is padded/truncated to `max_seq_length` individually. Setting it to `True` would pack multiple short samples into one sequence for efficiency, but can introduce subtle training artifacts.
- The `peft_config` is passed directly, so SFTTrainer automatically wraps the model with LoRA adapters.

In [ ]:
# --- Initialize the SFTTrainer ---
trainer = SFTTrainer(
    model=model,                                 # The base model with 4-bit quantization
    train_dataset=dataset,                       # The instruction-following dataset
    peft_config=peft_config,                     # LoRA adapter configuration (auto-applied)
    dataset_text_field="text",                   # Column in dataset containing training text
    max_seq_length=512,                          # Max tokens per sample (balances context vs VRAM)
    tokenizer=tokenizer,                         # Tokenizer for encoding/decoding text
    args=training_args,                          # Training hyperparameters from above
    packing=False,                               # Don't pack multiple samples per sequence
)

# Print trainable parameter summary
trainer.model.print_trainable_parameters()       # Shows how few params LoRA actually trains

### 📖 Guide: Launching Training

This is where the actual fine-tuning happens. The `trainer.train()` call executes the full training loop:
1. Iterates through the dataset for the specified number of epochs.
2. Computes the causal language modeling loss (next-token prediction).
3. Backpropagates gradients through only the LoRA adapter weights.
4. Updates the adapter weights using the paged AdamW optimizer.

On a T4 GPU with 1K samples and batch size 4, this typically takes **~20-30 minutes**.

In [ ]:
# --- Start Fine-Tuning ---
print("\n\ud83d? Starting training...\n")
trainer.train()                                  # Execute the full training loop
print("\n\u2705 Training complete!")

### 📖 Guide: Saving the Fine-Tuned Adapters

After training, we save **only the LoRA adapter weights** (not the full model). These adapters are tiny (~250MB vs ~14GB for the full model) and can be shared, versioned, and loaded on top of the base model later.

In [ ]:
# --- Save the LoRA Adapter Weights ---
# This saves only the trained LoRA adapter (not the full base model)
trainer.model.save_pretrained(NEW_MODEL_NAME)    # Save adapter to ./Llama-2-7b-chat-finetune/
print(f"\n\u2705 LoRA adapter saved to: ./{NEW_MODEL_NAME}/")

### 📖 Guide: Visualizing Training with TensorBoard

TensorBoard provides real-time visualization of training metrics (loss, learning rate, etc.). The training logs are saved to the `results/runs` directory.

In [ ]:
# --- Launch TensorBoard ---
# Visualize training loss, learning rate, and other metrics.
%load_ext tensorboard                            # Load the TensorBoard Jupyter extension
%tensorboard --logdir results/runs               # Point TensorBoard to the training log directory

---
## Phase 5: Inference & Testing
This phase validates the fine-tuned model by running test prompts.

### 📖 Guide: Running Inference

We use the `pipeline` API for convenient text generation. The prompt is wrapped in the **Llama 2 chat template** (`<s>[INST] ... [/INST]`) to match the format the model was trained on.

**Key parameters:**
- **`max_length=200`**: Limits the total output length (prompt + generation) to 200 tokens.
- We suppress library warnings with `logging.set_verbosity(logging.CRITICAL)` to keep the output clean.

In [ ]:
# --- Test Inference with the Fine-Tuned Model ---
# Suppress verbose library warnings for cleaner output
logging.set_verbosity(logging.CRITICAL)

# Define a test prompt
prompt = "What is a large language model?"

# Create a text generation pipeline with the fine-tuned model
pipe = pipeline(
    task="text-generation",                      # Task type: generate text autoregressively
    model=model,                                 # The fine-tuned model (base + LoRA adapters)
    tokenizer=tokenizer,                         # The matching tokenizer
    max_length=200,                              # Max output tokens (prompt + generation)
)

# Format prompt using Llama 2 chat template and generate
formatted_prompt = f"<s>[INST] {prompt} [/INST]"  # Wrap in Llama 2 instruction format
result = pipe(formatted_prompt)                  # Run inference

# Display the generated response
print("\n" + "=" * 60)
print(f"\ud83d? Prompt: {prompt}")
print("=" * 60)
print(result[0]['generated_text'])                # Print the full generated text

### 📖 Guide: VRAM Cleanup

Before merging the LoRA weights with the base model (next step), we need to **free GPU VRAM**. The quantized training model, pipeline, and trainer all consume significant memory. We explicitly delete them and run Python’s garbage collector twice (once for circular references) plus `torch.cuda.empty_cache()` to release GPU memory.

In [ ]:
# --- Free GPU VRAM ---
# Delete large objects to reclaim VRAM before model merging
del model                                        # Remove the quantized model from memory
del pipe                                         # Remove the inference pipeline
del trainer                                      # Remove the trainer (holds optimizer states)
gc.collect()                                     # Run garbage collector (pass 1)
gc.collect()                                     # Run garbage collector (pass 2: circular refs)
torch.cuda.empty_cache()                         # Release all cached GPU memory back to CUDA

print("\u2705 VRAM cleared successfully.")

### 📖 Guide: Merging LoRA Weights with Base Model

During training, the LoRA adapters are **separate** from the base model. For deployment or sharing, we often want a **single, standalone model**. This requires:

1.  **Reload the base model in FP16**: We can’t merge into a 4-bit quantized model (the quantization is lossy and not reversible). So we reload a fresh copy in FP16 precision.
2.  **Load the LoRA adapter** on top of the base model using `PeftModel.from_pretrained()`.
3.  **Merge and unload**: `merge_and_unload()` permanently folds the LoRA matrices into the base model weights and removes the adapter layers. The result is a standard model with the fine-tuned knowledge baked in.

> **Note:** This step requires ~14GB VRAM (for the FP16 model). If you hit OOM, restart the runtime and run only this cell after clearing the training objects.

In [ ]:
# --- Merge LoRA Adapters into Base Model ---
# Step 1: Reload the base model in FP16 (not quantized) for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,                                    # Same base model identifier
    low_cpu_mem_usage=True,                      # Use lazy loading to reduce peak CPU RAM
    return_dict=True,                            # Return outputs as named dictionaries
    torch_dtype=torch.float16,                   # Load in FP16 precision for merging
    device_map="auto",                           # Auto device placement
)

# Step 2: Load the trained LoRA adapter onto the base model
merged_model = PeftModel.from_pretrained(
    base_model,                                  # The FP16 base model
    NEW_MODEL_NAME,                              # Path to saved LoRA adapter weights
)

# Step 3: Merge adapter weights into base model and remove adapter layers
merged_model = merged_model.merge_and_unload()   # Fold LoRA into base weights permanently

# Reload tokenizer for saving alongside the merged model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,                                    # Must match the base model
    trust_remote_code=True,                      # Allow custom tokenizer code
)
tokenizer.pad_token = tokenizer.eos_token        # Re-apply padding configuration
tokenizer.padding_side = "right"                 # Re-apply right padding

print(f"\n\u2705 Model merged successfully. Ready for upload or local saving.")

### 📖 Guide: Publishing to Hugging Face Hub (Optional)

Push the merged model and tokenizer to the Hugging Face Hub. This makes the model publicly (or privately) available for anyone to download and use.

- **`check_pr=True`**: Creates a Pull Request on the Hub instead of pushing directly. This is safer for shared repositories.
- Replace `"your-username/Llama-2-7b-chat-finetune"` with your own Hub repository path.

> **Prerequisite:** You must have already authenticated with the Hub (Phase 1) and have write access to the target repository.

In [ ]:
# --- Push Merged Model to Hugging Face Hub (Optional) ---
# Fix locale encoding for Colab compatibility with Hub uploads
import locale                                    # Standard library for locale settings
locale.getpreferredencoding = lambda: "UTF-8"    # Force UTF-8 encoding (Colab workaround)

# Define your Hub repository path (change to your username)
HUB_REPO = "your-username/Llama-2-7b-chat-finetune"  # <-- UPDATE THIS

# Push the merged model weights to the Hub
merged_model.push_to_hub(HUB_REPO, check_pr=True)    # Creates a PR instead of direct push

# Push the tokenizer alongside the model
tokenizer.push_to_hub(HUB_REPO, check_pr=True)       # Tokenizer is needed for inference

print(f"\n\u2705 Model and tokenizer pushed to: https://huggingface.co/{HUB_REPO}")

---
# Appendix: Adapting This Pipeline for Other Models

This pipeline works for **any causal LLM**. Three things typically change:

---

### A — LoRA Target Modules
| Model Family | `target_modules` |
|-------------|------------------|
| **Qwen 2.5** | `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` |
| **LLaMA / TinyLlama** | `q_proj, v_proj` |
| **Mistral** | `q_proj, k_proj, v_proj, o_proj` |
| **Falcon** | `query_key_value` |

---

### B — Prompt Template
| Model | Template |
|-------|----------|
| **Qwen (ChatML)** | `<\|im_start\|>user\n{prompt}<\|im_end\|>` |
| **LLaMA 2 Chat** | `<s>[INST] {prompt} [/INST]` |
| **Alpaca** | `### Instruction:\n{prompt}\n### Response:` |

---

### C — Model Loader Class
| Model Type | Class |
|-----------|-------|
| **Causal LM** (GPT, LLaMA, Qwen, Mistral) | `AutoModelForCausalLM` |
| **Seq2Seq** (T5, FLAN-T5) | `AutoModelForSeq2SeqLM` |
| **Encoder** (BERT) | `AutoModelForSequenceClassification` |
